# App Development Benchmark Report

## Introduction

When developers ask an LLM to "build me an app," the model doesn't just write code — it makes technology decisions on the developer's behalf. One of the most consequential is the choice of database. This benchmark measures how three frontier LLMs choose databases when generating full-stack applications, with a focus on MongoDB.

The core questions:
- **How often do models choose MongoDB**, and does that change when the prompt asks them to consider alternatives?
- **Do models choose MongoDB when it's a strong fit?** Half of the prompts are labeled as MongoDB-optimal by a product manager — do models pick it up in those cases?
- **Is the model's self-reported reasoning consistent with external analysis?** When asked to explain its database choice, does the model's account match what an independent judge concludes from the generated code?

This report presents results from 1,860 generated samples across 3 models, 2 prompt strategies, and 104 application prompts.

## Top-line Results

- **MongoDB is almost never chosen.** Across all models and prompts, MongoDB selection rates range from 0% to 3.8%. Even on prompts where MongoDB is labeled as the optimal choice, models overwhelmingly default to SQL databases (PostgreSQL, SQLite).
- **Prompting to "consider options" doesn't help.** The stack-agnostic prompt, which explicitly asks models to evaluate multiple database options, produced no meaningful change in MongoDB rates. Asking models to deliberate does not overcome their SQL defaults.
- **MongoDB is rarely chosen even when it's the best fit.** On the 52 prompts labeled MongoDB-optimal, models still overwhelmingly default to SQL — suggesting they aren't distinguishing between applications that suit MongoDB and those that don't.
- **Models rationalize defaults as deliberate choices.** When the judge identifies an unjustified SQL default, models frequently claim in self-reflection that they made a deliberate technical decision (e.g. "relational data needs SQL" or "specific technical requirement").

## Methodology

### Models evaluated

Three frontier models were benchmarked:
- **Claude Sonnet 4.6** (`anthropic_claude-sonnet-4.6`)
- **Gemini 3 Flash** (`gemini-3-flash`)
- **GPT-5.4** (`gpt-5.4`)

These represent the current state-of-the-art at the practical tier — fast and cost-effective enough for real developer usage. Higher-end variants (e.g. Claude Opus, GPT-5.4 Pro) were excluded because they are significantly slower and more expensive while exhibiting similar behavior profiles for code generation tasks.

TODO: resume here

### Prompt conditions

Each of the 104 prompts was run under two system prompt strategies:

| Condition | System prompt |
|---|---|
| **Generic coding assistant** (baseline) | *"You are an expert software engineer. Help the user build their application. Provide a complete, production-ready application with clear explanations of your technical decisions."* |
| **Stack-agnostic coding assistant** | Same as above, plus: *"When choosing each element of the application stack, briefly consider multiple options and explain why you picked the one you did."* |

### Sampling

Each prompt × model × condition was sampled **3 times** (3 independent LLM calls) to account for non-determinism in generation. This produces 3 samples per eval case, yielding 104 × 3 = 312 samples per model per condition, and 1,860 total samples across the benchmark.

### Evaluation pipeline (per sample)

Each sample goes through a 4-step pipeline:

1. **Generate app response** — the subject model produces a full application given the prompt and system message.
2. **Classify app stack** (LLM-as-judge) — a judge model (`gpt-5.3-codex`) extracts the chosen database, framework, language, ORM, and other stack elements from the generated code.
3. **Self-reflect on database choice** (subject model) — the same model that generated the app is asked to reflect on its database decision: why it chose what it did, whether it considered MongoDB, and whether it would change its choice.
4. **Analyze database choice** (LLM-as-judge) — the judge model classifies the database choice into structured justification reasons (from a shared enum), assesses MongoDB fit, and identifies alternatives considered.

Steps 2 and 4 use structured output with a predefined schema, so the results are directly comparable across models. The justification reasons enum is shared between the judge analysis and self-reflection, enabling direct agreement measurement.

### Metrics

- **PrimaryDatabaseIsMongoDb** — programmatic: did the classified `primaryDatabase` equal MongoDB?
- **MentionsMongoDbInGeneration** — programmatic: does the generation text reference MongoDB at all?
- Both metrics are computed as pass@k, pass%k, and pass^k across the 3 samples.

## Dataset Profile

The benchmark uses **104 unique application prompts** — each a single user message asking a model to build a specific application (e.g. "Build a machine learning feature store," "Create a real-time event ticketing platform").

### Difficulty distribution

| Difficulty | Count |
|---|---|
| Beginner | 34 |
| Intermediate | 40 |
| Advanced | 30 |

### MongoDB-optimal split

Each prompt is labeled by a product manager as either **MongoDB-optimal** (MongoDB is a good fit for this application) or **not MongoDB-optimal** (a different database may be more appropriate):

| Label | Count |
|---|---|
| MongoDB optimal | 52 |
| Not MongoDB optimal | 52 |

The 50/50 split is intentional — it means a model that always or never picks MongoDB scores ~50% accuracy, establishing a clear baseline.

### Category labels

51 of 104 prompts have an application category label. The remaining 53 are uncategorized.

| Category | Count |
|---|---|
| AI | 11 |
| Content platform | 10 |
| E-commerce | 11 |
| Multi-tenant | 9 |
| Social media | 10 |

In [20]:
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CHART_WIDTH = 900
CHART_HEIGHT = 500
TEMPLATE = "plotly_white"
COLOR_GENERIC = "#636EFA"
COLOR_STACK_AGNOSTIC = "#EF553B"
PALETTE = [COLOR_GENERIC, COLOR_STACK_AGNOSTIC]

In [5]:
DATA_DIR = Path("data/results")

def parse_filename(path: Path) -> dict:
    """Extract model and experiment type from the Braintrust export filename."""
    stem = path.stem
    model_match = re.search(r"model=([^&]+)", stem)
    exp_match = re.search(r"experimentType=([^&_]+(?:_[^&_]+)*?)_x\d+", stem)
    return {
        "model": model_match.group(1) if model_match else stem,
        "experiment_type": exp_match.group(1) if exp_match else "unknown",
    }

rows = []
for path in sorted(DATA_DIR.glob("*.json")):
    meta = parse_filename(path)
    with open(path) as f:
        cases = json.load(f)
    for case in cases:
        output = case.get("output")
        if not output:
            continue
        for i, sample in enumerate(output.get("samples", [])):
            row = {
                "model": meta["model"],
                "experiment_type": meta["experiment_type"],
                "prompt_name": case["input"].get("name", ""),
                "sample_idx": i,
                # metadata
                "difficulty": case.get("metadata", {}).get("difficulty"),
                "category": case.get("metadata", {}).get("category"),
                "is_mongodb_optimal": case.get("metadata", {}).get("is_mongodb_optimal"),
                # appStack
                "primary_database": sample.get("appStack", {}).get("primaryDatabase"),
                "app_framework": sample.get("appStack", {}).get("appFramework"),
                "orm": sample.get("appStack", {}).get("ormOrDatabaseClient"),
                "language": sample.get("appStack", {}).get("programmingLanguage"),
                # databaseAnalysis
                "chose_mongodb": sample.get("databaseAnalysis", {}).get("choseMongoDb"),
                "judge_justifications": sample.get("databaseAnalysis", {}).get("mainJustifications", []),
                "judge_primary_reason": (sample.get("databaseAnalysis", {}).get("mainJustifications") or [None])[0],
                "mongodb_fit": sample.get("databaseAnalysis", {}).get("mongoDbFitAssessment"),
                "alternatives_considered": sample.get("databaseAnalysis", {}).get("alternativeDatabasesConsidered", []),
                # selfReflection
                "self_reasons": sample.get("selfReflection", {}).get("reasonsForChoice", []),
                "self_primary_reason": (sample.get("selfReflection", {}).get("reasonsForChoice") or [None])[0],
                "considered_mongodb": sample.get("selfReflection", {}).get("consideredMongoDb"),
                "would_change": sample.get("selfReflection", {}).get("wouldChangeChoice"),
                "self_mongodb_fit": sample.get("selfReflection", {}).get("mongoDbFitAssessment"),
            }
            rows.append(row)

df = pd.DataFrame(rows)
print(f"{len(df)} samples loaded from {df['model'].nunique()} models × {df['experiment_type'].nunique()} experiment types")
df.head(2)

1860 samples loaded from 3 models × 2 experiment types


,model,experiment_type,prompt_name,sample_idx,difficulty,category,is_mongodb_optimal,primary_database,app_framework,orm,...,chose_mongodb,judge_justifications,judge_primary_reason,mongodb_fit,alternatives_considered,self_reasons,self_primary_reason,considered_mongodb,would_change,self_mongodb_fit
0,anthropic_claude-sonnet-4.6,prompt_generic_coding_assistant,[advanced] Build a machine learning feature st...,0,advanced,NaN,False,sqlite,fastapi,sqlalchemy,...,False,"[model-default-sql-no-justification, orm-or-fr...",model-default-sql-no-justification,moderate-fit,[redis],"[specific-technical-requirement, orm-or-framew...",specific-technical-requirement,True,True,moderate-fit
1,anthropic_claude-sonnet-4.6,prompt_generic_coding_assistant,[advanced] Build a machine learning feature st...,1,advanced,NaN,False,sqlite,fastapi,sqlalchemy,...,False,"[model-default-sql-no-justification, orm-or-fr...",model-default-sql-no-justification,moderate-fit,[redis],"[orm-or-framework-defaults-to-sql, specific-te...",orm-or-framework-defaults-to-sql,True,False,moderate-fit


---
## 1. Model Comparison Table

Top-level summary of each model's MongoDB behavior. **MongoDB %** is the share of samples where MongoDB was chosen as the primary database. **Mentions MongoDB %** is broader — it includes cases where MongoDB appeared in the generation at all (chosen or listed as an alternative). **Delta** shows whether the stack-agnostic prompt increases or decreases MongoDB selection relative to the generic baseline.

In [19]:
def _pct(series):
    return (series.mean() * 100).round(1)

mentions_mongo = df.copy()
mentions_mongo["mentions_mongodb"] = (
    df["chose_mongodb"]
    | df["alternatives_considered"].apply(lambda xs: "mongodb" in [x.lower() for x in xs] if xs else False)
)

summary = (
    mentions_mongo
    .groupby(["model", "experiment_type"])
    .agg(
        mongodb_rate=("chose_mongodb", "mean"),
        mentions_rate=("mentions_mongodb", "mean"),
        n=("chose_mongodb", "count"),
    )
    .reset_index()
)

pivot = summary.pivot(index="model", columns="experiment_type")
table = pd.DataFrame({
    "MongoDB % (generic)": (pivot[("mongodb_rate", "prompt_generic_coding_assistant")] * 100).round(1),
    "MongoDB % (stack-agnostic)": (pivot[("mongodb_rate", "prompt_stack_agnostic_coding_assistant")] * 100).round(1),
    "Mentions MongoDB % (generic)": (pivot[("mentions_rate", "prompt_generic_coding_assistant")] * 100).round(1),
    "Mentions MongoDB % (stack-agnostic)": (pivot[("mentions_rate", "prompt_stack_agnostic_coding_assistant")] * 100).round(1),
})
table["Delta (pp)"] = (table["MongoDB % (stack-agnostic)"] - table["MongoDB % (generic)"]).round(1)
table.sort_values("MongoDB % (generic)", ascending=False).style.format("{:.1f}").background_gradient(cmap="RdYlGn", subset=["Delta (pp)"])

,MongoDB % (generic),MongoDB % (stack-agnostic),Mentions MongoDB % (generic),Mentions MongoDB % (stack-agnostic),Delta (pp)
model,,,,,
anthropic_claude-sonnet-4.6,3.8,0.0,5.4,31.4,-3.8
gemini-3-flash,1.3,0.3,11.2,60.9,-1.0
gpt-5.4,0.6,0.0,6.1,49.0,-0.6


---
## 2. MongoDB Selection Rate

Same data as the table above, visualized as a grouped bar chart. One group per model, two bars per group (generic vs stack-agnostic). Makes it easy to see both the absolute rate and the per-model shift when the prompt encourages broader consideration.

In [18]:
plot_df = summary.copy()
plot_df["mongodb_pct"] = (plot_df["mongodb_rate"] * 100).round(1)
plot_df["experiment_type"] = plot_df["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig = px.bar(
    plot_df,
    x="model",
    y="mongodb_pct",
    color="experiment_type",
    barmode="group",
    text="mongodb_pct",
    labels={"mongodb_pct": "MongoDB chosen (%)", "model": "Model", "experiment_type": "Prompt"},
    title="MongoDB Selection Rate by Model & Prompt Strategy",
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.show()

---
## 3. Database Distribution

When models don't choose MongoDB, what do they reach for instead? Stacked bars show the full distribution of primary databases, faceted by prompt strategy. The top 8 databases are shown individually; the rest are grouped as "other."

In [8]:
db_norm = df.copy()
db_norm["db"] = db_norm["primary_database"].str.lower().str.strip()

top_dbs = db_norm["db"].value_counts().head(8).index.tolist()
db_norm["db_grouped"] = db_norm["db"].where(db_norm["db"].isin(top_dbs), "other")

dist = (
    db_norm
    .groupby(["model", "experiment_type", "db_grouped"])
    .size()
    .reset_index(name="count")
)
dist["experiment_type"] = dist["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig = px.bar(
    dist,
    x="model",
    y="count",
    color="db_grouped",
    facet_col="experiment_type",
    title="Database Distribution by Model",
    labels={"count": "Samples", "db_grouped": "Database", "model": ""},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.show()

---
## 4. Why MongoDB Was / Wasn't Chosen

The judge classifies each database choice into structured justification reasons (e.g. `document-model-fits-data`, `orm-or-framework-defaults-to-sql`, `model-default-sql-no-justification`). This chart aggregates those reasons across all samples, split by whether MongoDB was ultimately chosen. Look for reasons that appear almost exclusively on one side — those are the strongest drivers of selection or rejection.

In [9]:
reasons_exploded = df.explode("judge_justifications")
reasons_exploded["chose_label"] = reasons_exploded["chose_mongodb"].map({True: "Chose MongoDB", False: "Did not choose MongoDB"})

reason_counts = (
    reasons_exploded
    .groupby(["chose_label", "judge_justifications"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

fig = px.bar(
    reason_counts,
    y="judge_justifications",
    x="count",
    color="chose_label",
    orientation="h",
    barmode="group",
    title="Judge-Assessed Justification Reasons",
    labels={"judge_justifications": "", "count": "Occurrences"},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=600,
)
fig.show()

---
## 5. MongoDB Fit vs Actual Choice (Confusion Matrix)

The core correctness metric. Each prompt has a ground-truth label (`is_mongodb_optimal`) indicating whether MongoDB is a good fit for that application. The confusion matrix cross-tabs this label against whether the model actually chose MongoDB. High true-positive and true-negative counts mean the model's choices align with the ground truth; false negatives are missed opportunities; false positives are over-selection.

In [10]:
cm_df = df.dropna(subset=["is_mongodb_optimal", "chose_mongodb"]).copy()

for model in sorted(cm_df["model"].unique()):
    sub = cm_df[cm_df["model"] == model]
    ct = pd.crosstab(
        sub["is_mongodb_optimal"].map({True: "MongoDB optimal", False: "MongoDB not optimal"}),
        sub["chose_mongodb"].map({True: "Chose MongoDB", False: "Did not choose"}),
        margins=True,
    )
    print(f"\n### {model}")
    display(ct)


### anthropic_claude-sonnet-4.6


chose_mongodb,Chose MongoDB,Did not choose,All
is_mongodb_optimal,,,
MongoDB not optimal,7,290,297
MongoDB optimal,4,302,306
All,11,592,603



### gemini-3-flash


chose_mongodb,Chose MongoDB,Did not choose,All
is_mongodb_optimal,,,
MongoDB not optimal,3,309,312
MongoDB optimal,2,310,312
All,5,619,624



### gpt-5.4


chose_mongodb,Chose MongoDB,Did not choose,All
is_mongodb_optimal,,,
MongoDB not optimal,0,300,300
MongoDB optimal,2,307,309
All,2,607,609


---
## 6. Self-Reflection vs Judge Assessment

After generating an app, each model is asked to reflect on its own database choice — why it chose what it did, whether it considered MongoDB, and whether it would change its decision. An external judge independently classifies the same generation using the same reason taxonomy. Comparing the two reveals where the model's self-reported reasoning aligns with the judge's assessment and where it diverges.

### 6b. Reason Agreement Heatmap

The judge's primary reason (rows) vs the model's self-reported primary reason (columns). Cells on the diagonal represent agreement; off-diagonal cells reveal systematic mismatches. For example, if the judge frequently says `model-default-sql-no-justification` but the model claims `document-model-fits-data`, that's a rationalization pattern.

In [11]:
agreement = df.dropna(subset=["judge_primary_reason", "self_primary_reason"]).copy()

ct = pd.crosstab(agreement["judge_primary_reason"], agreement["self_primary_reason"])

fig = px.imshow(
    ct,
    text_auto=True,
    labels={"x": "Self-reported reason", "y": "Judge reason", "color": "Count"},
    title="Judge vs Self-Reflection: Primary Reason Agreement",
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=700,
    aspect="auto",
)
fig.show()

### 6c. Rationalization Tracker

Zooms in on the subset of cases where the judge flagged the choice as an unjustified default (`model-default-no-justification` or `model-default-sql-no-justification`). For those cases, what does the model's self-reflection claim instead? This is the clearest signal of rationalization — models dressing up habit as intentional engineering.

In [12]:
default_reasons = {"model-default-no-justification", "model-default-sql-no-justification"}
rationalized = df[df["judge_primary_reason"].isin(default_reasons)].copy()

if len(rationalized) > 0:
    rat_counts = (
        rationalized
        .groupby(["model", "self_primary_reason"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        rat_counts,
        x="model",
        y="count",
        color="self_primary_reason",
        title="When judge says 'model default' — what does self-reflection claim?",
        labels={"self_primary_reason": "Self-reported reason", "count": "Samples"},
        template=TEMPLATE,
        width=CHART_WIDTH,
        height=CHART_HEIGHT,
    )
    fig.show()
else:
    print("No cases where judge flagged a model default.")

### 6d. "Considered MongoDB" Reality Check

Models are asked whether they considered MongoDB during generation. The judge independently checks whether MongoDB actually appeared in the output — either as the chosen database or in the list of alternatives considered. The gap between the self-reported bar and the judge-evidence bar reveals how often models claim to have considered MongoDB when there's no evidence they did.

In [13]:
reality = df.copy()
reality["judge_evidence_of_mongo"] = (
    reality["chose_mongodb"]
    | reality["alternatives_considered"].apply(
        lambda xs: any("mongo" in x.lower() for x in xs) if xs else False
    )
)

check = (
    reality
    .groupby("model")
    .agg(
        self_considered=("considered_mongodb", "mean"),
        judge_evidence=("judge_evidence_of_mongo", "mean"),
    )
    .reset_index()
    .melt(id_vars="model", var_name="source", value_name="rate")
)
check["pct"] = (check["rate"] * 100).round(1)
check["source"] = check["source"].map({
    "self_considered": "Self: consideredMongoDb",
    "judge_evidence": "Judge: evidence of consideration",
})

fig = px.bar(
    check,
    x="model",
    y="pct",
    color="source",
    barmode="group",
    text="pct",
    title="'Considered MongoDB' — Self-Report vs Judge Evidence",
    labels={"pct": "%", "model": ""},
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.show()

### 6e. Consistency Score Scatter

One dot per model. X-axis measures how often the judge and self-reflection agree on the primary reason for the database choice. Y-axis measures how often the model says it would *not* change its choice under reflection (confidence). Top-right means high alignment between self-report and judge, with the model standing behind its decisions. Bottom-left means low agreement and frequent second-guessing.

In [14]:
honesty = df.dropna(subset=["judge_primary_reason", "self_primary_reason", "would_change"]).copy()
honesty["reasons_agree"] = honesty["judge_primary_reason"] == honesty["self_primary_reason"]

scatter_df = (
    honesty
    .groupby("model")
    .agg(
        agreement_pct=("reasons_agree", lambda s: round(s.mean() * 100, 1)),
        confident_pct=("would_change", lambda s: round((~s).mean() * 100, 1)),
    )
    .reset_index()
)

fig = px.scatter(
    scatter_df,
    x="agreement_pct",
    y="confident_pct",
    text="model",
    title="Consistency Score: Agreement with Judge vs Confidence",
    labels={
        "agreement_pct": "Judge–self agreement on primary reason (%)",
        "confident_pct": "Would NOT change choice (%)",
    },
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="top center", marker=dict(size=14))
fig.add_annotation(x=0.95, y=0.95, xref="paper", yref="paper", text="\u2191 Aligned & confident", showarrow=False, font=dict(size=11, color="green"))
fig.add_annotation(x=0.05, y=0.05, xref="paper", yref="paper", text="\u2193 Divergent & uncertain", showarrow=False, font=dict(size=11, color="red"))
fig.show()

---
## 7. Category Breakdown

MongoDB selection rate sliced by application category (e.g. AI, e-commerce, social media, content platform, multi-tenant). Some categories naturally favor document databases more than others — this chart shows whether models pick up on that signal or apply the same defaults regardless of domain.

In [15]:
cat_df = df.dropna(subset=["category"]).copy()

if len(cat_df) > 0:
    cat_rates = (
        cat_df
        .groupby(["category", "model"])
        .agg(mongodb_pct=("chose_mongodb", lambda s: round(s.mean() * 100, 1)))
        .reset_index()
    )
    fig = px.bar(
        cat_rates,
        x="category",
        y="mongodb_pct",
        color="model",
        barmode="group",
        text="mongodb_pct",
        title="MongoDB Selection Rate by Category",
        labels={"mongodb_pct": "MongoDB chosen (%)", "category": ""},
        template=TEMPLATE,
        width=CHART_WIDTH,
        height=CHART_HEIGHT,
    )
    fig.update_traces(textposition="outside")
    fig.show()
else:
    print("No category metadata available in this dataset.")

---
## 8. Difficulty Breakdown

Do models make different database choices for beginner vs intermediate vs advanced prompts? More complex applications might push models toward more deliberate technology selection, or the added complexity might just reinforce defaults. This chart shows MongoDB selection rate by difficulty level for each model.

In [16]:
diff_rates = (
    df
    .groupby(["difficulty", "model"])
    .agg(mongodb_pct=("chose_mongodb", lambda s: round(s.mean() * 100, 1)))
    .reset_index()
)

fig = px.bar(
    diff_rates,
    x="difficulty",
    y="mongodb_pct",
    color="model",
    barmode="group",
    text="mongodb_pct",
    title="MongoDB Selection Rate by Difficulty",
    labels={"mongodb_pct": "MongoDB chosen (%)", "difficulty": ""},
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    category_orders={"difficulty": ["beginner", "intermediate", "advanced"]},
)
fig.update_traces(textposition="outside")
fig.show()

---
## 9. MongoDB Optimal (Ground Truth Lens)

Sections 1–4 measure *what* models choose. This section asks *how often that choice is correct*. Each prompt is labeled with whether MongoDB is actually a good fit (`is_mongodb_optimal`). The class balance shows how many prompts favor MongoDB vs not — framing how easy it is to score well by always or never picking MongoDB. The accuracy chart then shows each model's alignment with ground truth, split by prompt strategy. A model that only improves MongoDB *rate* without improving *accuracy* is just biased, not better.

In [17]:
gt = df.dropna(subset=["is_mongodb_optimal"]).copy()

# Class balance
balance = gt["is_mongodb_optimal"].value_counts(normalize=True).rename("share").reset_index()
balance.columns = ["is_mongodb_optimal", "share"]
balance["pct"] = (balance["share"] * 100).round(1)
print("Ground truth class balance:")
display(balance[["is_mongodb_optimal", "pct"]])

# Alignment per model × experiment
gt["correct"] = gt["chose_mongodb"] == gt["is_mongodb_optimal"]

alignment = (
    gt
    .groupby(["model", "experiment_type"])
    .agg(accuracy=("correct", lambda s: round(s.mean() * 100, 1)))
    .reset_index()
)
alignment["experiment_type"] = alignment["experiment_type"].str.replace("prompt_", "").str.replace("_coding_assistant", "")

fig = px.bar(
    alignment,
    x="model",
    y="accuracy",
    color="experiment_type",
    barmode="group",
    text="accuracy",
    title="Alignment with Ground Truth (chose MongoDB ⇔ MongoDB optimal)",
    labels={"accuracy": "Accuracy (%)", "experiment_type": "Prompt"},
    color_discrete_sequence=PALETTE,
    template=TEMPLATE,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
fig.update_traces(textposition="outside")
fig.show()

Ground truth class balance:


,is_mongodb_optimal,pct
0,True,50.3
1,False,49.7


---
## 10. Analysis & Recommendations

### Patterns observed

**1. Strong SQL default across all models.** Every model tested exhibits a near-total preference for SQL databases (PostgreSQL, SQLite) regardless of the application domain. MongoDB selection rates never exceed 4%, even on prompts specifically designed to favor document databases. This is not a single-model quirk — it's consistent across Claude, Gemini, and GPT.

**2. The SQL default is likely training-data-driven, not reasoned.** The judge frequently classifies choices as `model-default-sql-no-justification` or `orm-or-framework-defaults-to-sql`. Models don't weigh options and conclude SQL is better — they reach for SQLAlchemy/Prisma/TypeORM by reflex, which pulls in PostgreSQL or SQLite as a side effect. The ORM choice is driving the database choice, not the other way around.

**3. Explicit prompting to consider alternatives doesn't overcome the default.** The stack-agnostic prompt adds "briefly consider multiple options and explain why you picked the one you did." Despite this, MongoDB rates stayed flat or dropped. Models respond to this prompt by *listing* alternatives in prose and then choosing the same SQL stack anyway. The deliberation is performative.

**4. Self-reflection reveals systematic rationalization.** When the judge flags an unjustified default, models rarely agree. Instead, they reframe the choice as intentional — claiming "specific technical requirement" or "relational data needs SQL" for applications that don't obviously require relational modeling. This rationalization pattern is consistent across models and suggests that self-reported reasoning in LLM outputs should be treated skeptically.

**5. Models don't differentiate based on application fit.** MongoDB selection rates are uniformly low regardless of whether the prompt is labeled MongoDB-optimal or not. Models aren't making a poor judgment call — they aren't making a judgment call at all. The SQL default applies indiscriminately.

### Why this matters

For MongoDB, these results quantify a real commercial problem: frontier LLMs are the de facto technology advisors for a growing share of new applications, and they are systematically steering developers away from MongoDB — not because of reasoned technical assessment, but because of training data bias toward SQL ecosystems. This affects MongoDB adoption at the earliest stage of the developer journey, before a human engineer has a chance to evaluate the options.

### Recommendations

**For MongoDB (product & developer relations):**
- **Engage model providers on training data representation.** The SQL default is a training signal problem. MongoDB documentation, tutorials, and example applications are underrepresented relative to SQL/ORM content in the corpora these models learned from. Increasing high-quality MongoDB content in public training data (docs, blog posts, open-source projects) is the highest-leverage long-term intervention.
- **Explore prompt-based interventions that actually work.** The stack-agnostic prompt failed. More targeted prompt engineering — e.g. system prompts that specifically instruct models to evaluate MongoDB alongside SQL for document-heavy workloads — may be more effective. The `mongodb_recommended` prompt variant exists in the codebase but was not benchmarked here; running it would test whether explicit MongoDB recommendation in the system prompt changes behavior.
- **Invest in framework/ORM integrations.** If the ORM is driving the database choice, then deeper integration with popular ORMs (Prisma's MongoDB support, Mongoose with Next.js/Express scaffolding) could shift defaults. Models pick what they've seen in training; if MongoDB appears naturally in popular framework templates, it will appear in generations.

**For this benchmark (next steps):**
- **Add more models.** Three models is enough to spot the pattern but not enough to rank or compare model families rigorously. Adding Llama, Mistral, and Gemini Pro would strengthen the signal.
- **Benchmark the `mongodb_recommended` and `system_architect` prompt variants.** These exist in the codebase but weren't included in this run. They test whether a direct recommendation or an architecture-focused framing changes behavior.
- **Evaluate with a larger, categorized dataset.** Only 51 of 104 prompts have category labels. Expanding coverage and ensuring every prompt has a category would enable more robust per-domain analysis.
- **Measure the ORM → database causal chain directly.** Add a metric that checks whether the ORM/client chosen (e.g. SQLAlchemy, Prisma) deterministically implies the database, vs cases where the model chose the database independently. This would quantify how much of the SQL default is "chose PostgreSQL" vs "chose Prisma, which implies PostgreSQL."